# SciGraphAgent Benchmark — Notebook 06
## Building an Interactive Dashboard with Streamlit

---

**Notebook series:**
- Notebook 00 — Setup and foundations ✓
- Notebook 01 — Loading benchmark datasets ✓
- Notebook 02 — Building retrieval systems ✓
- Notebook 03 — Running experiments ✓
- Notebook 04 — Computing metrics ✓
- Notebook 05 — Visualising results ✓
- **Notebook 06 — Interactive dashboard ← you are here**

---

## What this notebook teaches

**Part A — Practical constraints:**
- Why dashboards exist — what they add over static figures
- Streamlit vs Flask vs Dash — which to use and when
- Why step06 costs $0 — reads from JSON, no API calls
- How Streamlit's execution model works — why the script reruns on every interaction
- What `@st.cache_data` does and why it is essential for performance
- Plotly vs matplotlib — when interactive beats static
- Deprecation warnings — what they mean and how to fix them

**Part B — Dashboard concepts:**
- What a sidebar is and why it separates controls from content
- What sliders do — live threshold adjustment without re-running experiments
- What metric cards are — KPI display for quick reading
- What a drill-down table is — from aggregate to individual questions
- How HTML in Streamlit works — condition colour cards
- How `st.stop()` works — graceful error handling

**Part C — Running step06:**
- Launch the dashboard and explore all six sections
- Adjust thresholds and observe how gate verdicts change
- Understand what each section shows at n=3 vs n=50
- Pre-commit verification and commit

---

## Important: this notebook explains a Streamlit app

Unlike previous notebooks, step06 **cannot run inside a Jupyter notebook cell**. Streamlit is a web application framework — it starts a local web server and opens in your browser. The notebook cells here:
1. Explain the concepts and code
2. Verify the dashboard file exists and is valid
3. Show you how to launch and use it
4. Demonstrate the individual Streamlit components in isolation

The actual dashboard runs with: `streamlit run step06_dashboard.py`

---
## Section 1 — Environment Setup

In [ ]:
from dotenv import load_dotenv
import os, json, subprocess
from pathlib import Path

cwd = Path(os.getcwd())
dotenv_path = cwd.parent / ".env" if (cwd.parent / ".env").exists() else cwd / ".env"
load_dotenv(dotenv_path)

print("Working directory:", cwd)
print("API calls needed : ZERO (step06 reads JSON files only)")
print("Cost             : $0.00")
print()

# Check prerequisites
required = [
    Path("step06_dashboard.py"),
    Path("results") / "metrics_hotpotqa_3.json",
    Path("results") / "raw_results_hotpotqa_3.json",
]
all_ok = True
for p in required:
    exists = p.exists()
    size   = f"{p.stat().st_size // 1024} KB" if exists else ""
    print(f"  {'✓' if exists else '✗'} {p}  {size}")
    if not exists: all_ok = False

# Check streamlit is installed
print()
try:
    import streamlit as st
    print(f"  ✓ streamlit {st.__version__} installed")
except ImportError:
    print("  ✗ streamlit not installed — run: pip install streamlit")
    all_ok = False

try:
    import plotly
    print(f"  ✓ plotly {plotly.__version__} installed")
except ImportError:
    print("  ✗ plotly not installed — run: pip install plotly")
    all_ok = False

print()
print("✓ Ready" if all_ok else "✗ Fix issues above first")
print()
print("To launch the dashboard:")
print("  streamlit run step06_dashboard.py")
print("  Then open http://localhost:8501 in your browser")

---
# PART A — Practical Constraints

## Section 2 — Why Dashboards Exist

### What static figures cannot do

The five PNG figures from step05 are excellent for a paper — they are precisely sized, 300 dpi, and communicate one fixed argument each. But they have a fundamental limitation: **they are frozen at the moment of creation**.

A reviewer reading your paper cannot:
- Change the faithfulness threshold from 0.75 to 0.80 and see how the gate verdict changes
- Switch from HotpotQA to MuSiQue and see the same charts
- Click on Condition D bar to see which specific questions it answered correctly
- Ask "what happens if I set the recall lift target to +20pp instead of +15pp?"

A dashboard answers all of these questions interactively.

### What dashboards add

| Static figure (step05) | Interactive dashboard (step06) |
|---|---|
| Fixed at creation time | Updates on every interaction |
| One dataset, one n | Switch datasets and n in sidebar |
| Fixed thresholds | Adjustable thresholds via sliders |
| Aggregate only | Drill down to individual questions |
| For paper/PDF | For exploration and demos |
| 300 dpi PNG | Interactive HTML/web |

### When to build a dashboard vs static figures

Build **both**. They serve different audiences:
- Static figures → paper reviewers, published PDF
- Dashboard → collaborators, demo audience, your own exploration during development

The dashboard is also your development tool — you can adjust thresholds and immediately see the effect without re-running any experiments.

---
## Section 3 — Streamlit vs Flask vs Dash: Which to Use

### The three main Python web frameworks for data apps

**Flask** — a general-purpose web framework. You write HTML templates, handle HTTP routes, manage sessions. Maximum flexibility, maximum complexity. Learning curve: weeks. Good for production APIs and complex web apps.

**Dash** (by Plotly) — built for data dashboards. Declarative layout system, callback functions for interactivity. More structured than Flask, less magic than Streamlit. Good for complex dashboards with precise layout control.

**Streamlit** — designed specifically for ML engineers and data scientists. Write a Python script, it becomes a web app. No HTML, no JavaScript, no callbacks. Learning curve: hours.

### Why we chose Streamlit

| Requirement | Streamlit | Dash | Flask |
|---|---|---|---|
| pip install only | ✓ | ✓ | ✓ |
| No HTML/JS needed | ✓ | ~ | ✗ |
| Works in 1 file | ✓ | ~ | ✗ |
| Interactive charts | ✓ (Plotly) | ✓ (Plotly) | ~ |
| Sidebar controls | ✓ built-in | ~ custom | ✗ |
| Slider widgets | ✓ 1 line | ✓ 5 lines | ✗ |
| Learning curve | Hours | Days | Weeks |

For a benchmark results dashboard built by one researcher, Streamlit is the right tool.

### How Streamlit's execution model works

This is the most important thing to understand about Streamlit — and the thing that surprises every beginner.

**Every time the user interacts with any widget (slider, dropdown, button), the entire Python script reruns from top to bottom.**

This is different from every other framework. In Flask or Dash, you write callback functions that run only when a specific widget changes. In Streamlit, everything reruns.

**Why?** Simplicity. You write the script once, top to bottom, as if it runs once. Streamlit handles all the state management invisibly.

**The problem:** If your script loads a 1 GB file at the top, every slider interaction reloads the file — making the app unbearably slow.

**The solution:** `@st.cache_data` — shown in the next section.

In [ ]:
# Demonstrate Streamlit's execution model conceptually
# (cannot run Streamlit inside Jupyter — this is a conceptual demo)

import time

print("Streamlit execution model — conceptual demonstration")
print("=" * 55)
print()
print("Without @st.cache_data:")
print()

def load_results_slow(path):
    """Simulates slow file loading (imagine a 100 MB file)."""
    import json
    from pathlib import Path
    t0 = time.perf_counter()
    with open(path) as f:
        data = json.load(f)
    elapsed = time.perf_counter() - t0
    return data, elapsed

from pathlib import Path
path = Path("results") / "metrics_hotpotqa_3.json"

# Simulate 3 slider interactions — each triggers a full script rerun
total_load_time = 0
for interaction in range(1, 4):
    data, elapsed = load_results_slow(path)
    total_load_time += elapsed
    print(f"  Slider interaction {interaction}: file reloaded in {elapsed*1000:.1f}ms")

print(f"  Total file load time across 3 interactions: {total_load_time*1000:.1f}ms")
print()
print("For an 18KB file this is fine. For a 100MB results file at n=1000:")
print("  Each load would take ~500ms → dashboard feels sluggish")
print()
print("With @st.cache_data:")
print("  First load: ~500ms (reads from disk)")
print("  Subsequent loads: ~0ms (returns cached result)")
print("  3 interactions: ~500ms total (not 1500ms)")
print()
print("This is why every file-loading function in step06 has @st.cache_data.")
print()
print("In step06_dashboard.py:")
print("  @st.cache_data")
print("  def load_metrics(dataset, n):")
print("      path = RESULTS_DIR / f'metrics_{dataset}_{n}.json'")
print("      with open(path) as f:")
print("          return json.load(f)")
print()
print("The decorator caches the return value. Calling load_metrics('hotpotqa', 3)")
print("twice returns the same object — the file is only read once per session.")

### 📌 Retain: Streamlit's execution model and caching

**General principle:** In Streamlit, every widget interaction reruns the entire script. Without caching, expensive operations (file I/O, API calls, model inference) repeat on every interaction.

**The fix — one decorator:**
```python
@st.cache_data
def load_expensive_data(param):
    # This runs only once per unique (param) value
    return expensive_operation(param)
```

**Three caching decorators to know:**
- `@st.cache_data` — for data (DataFrames, dicts, strings). Returns a copy each call.
- `@st.cache_resource` — for resources (database connections, ML models). Returns the same object.
- No decorator — reruns every time (correct for cheap operations like computing a sum)

**Generalise:** This pattern — cache expensive operations, recompute cheap ones — applies to any reactive UI framework: React (useMemo), Vue (computed), Dash (callback caching). The principle is always the same: identify what is expensive, cache it.

---
## Section 4 — Plotly vs Matplotlib: When Interactive Beats Static

### The key difference

Matplotlib renders a figure and produces a static image (PNG, PDF, SVG). Once rendered, it is frozen — a picture.

Plotly renders a figure as an interactive HTML/JavaScript object. The viewer can:
- Hover over any bar to see exact values
- Click legend items to show/hide traces
- Zoom into a region of interest
- Pan across the chart
- Download a PNG snapshot

### When to use each

| Scenario | Use |
|---|---|
| Paper figure (PDF submission) | matplotlib (step05) |
| Dashboard chart | Plotly (step06) |
| Quick EDA in a notebook | Either |
| Server-side rendering (no browser) | matplotlib |
| Hover tooltips needed | Plotly |

### Deprecation warning we fixed

When you first ran the dashboard, Streamlit printed:
```
Please replace `use_container_width` with `width`.
`use_container_width` will be removed after 2025-12-31.
```

**What this means:** Streamlit 1.62.0 deprecated the `use_container_width=True` parameter. The new API uses `width='stretch'` for the same effect. This is a **deprecation warning**, not an error — the app still works, but will break when `use_container_width` is removed in a future version.

**We fixed it** by replacing all instances of `use_container_width=True` with `width='stretch'` using VS Code's global find-and-replace (`Ctrl+H`).

**Lesson:** Always read deprecation warnings. They tell you exactly what will break and when, giving you time to fix it before it becomes an error.

In [ ]:
# Demonstrate Plotly chart creation — the same pattern used in step06
# This renders inline in the notebook as a static image
# (full interactivity only works in the Streamlit browser app)

import plotly.graph_objects as go
import json
from pathlib import Path

# Load real metrics
with open(Path("results") / "metrics_hotpotqa_3.json") as f:
    metrics = json.load(f)

# Build the same paired bar chart as in step06 Section 1
e1 = metrics.get("exp1", {})
ng = e1.get("no_gate", {})
wg = e1.get("with_gate", {})

metric_labels = ["Faithfulness", "F1 Score", "Relevancy"]
metric_keys   = ["faithfulness", "f1",       "relevancy"]

fig = go.Figure()

# Trace 1: no-gate bars
fig.add_trace(go.Bar(
    name="No gate (single pass)",
    x=metric_labels,
    y=[ng.get(k, 0) for k in metric_keys],
    marker_color="#56B4E9",         # sky blue
    text=[f"{ng.get(k,0):.3f}" for k in metric_keys],
    textposition="outside",
))

# Trace 2: with-gate bars
fig.add_trace(go.Bar(
    name="With gate (RAGAS retry)",
    x=metric_labels,
    y=[wg.get(k, 0) for k in metric_keys],
    marker_color="#009E73",         # green
    text=[f"{wg.get(k,0):.3f}" for k in metric_keys],
    textposition="outside",
))

# Horizontal threshold line
fig.add_hline(
    y=0.75,
    line_dash="dot",
    line_color="#CC79A7",
    annotation_text="Faith threshold (0.75)",
    annotation_position="right",
)

fig.update_layout(
    title="Retry Gate vs Single Pass — Demo (HOTPOTQA, n=3)",
    barmode="group",
    yaxis=dict(range=[0, 1.2], title="Score (0–1)"),
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)

fig.show()   # in notebook: static. In Streamlit: interactive

print()
print("In the Streamlit dashboard, this chart supports:")
print("  - Hover over bars to see exact values")
print("  - Click legend to hide/show 'No gate' or 'With gate'")
print("  - Zoom by drawing a selection box")
print("  - Download PNG via the camera icon (top right)")

### 📌 Retain: Plotly for interactive dashboards

**General principle:** Use Plotly for interactive dashboards, matplotlib for static publication figures. Both use the same data — the figure type is a presentation choice, not a data choice.

**Plotly figure pattern:**
```python
fig = go.Figure()
fig.add_trace(go.Bar(x=categories, y=values, name="label"))
fig.add_hline(y=threshold, line_dash="dot")  # threshold line
fig.update_layout(title="...", yaxis=dict(range=[0, 1.2]))
st.plotly_chart(fig, width='stretch')  # in Streamlit
fig.show()                              # in Jupyter
```

**Deprecation warnings — always fix them:**
When a library prints a deprecation warning, fix it immediately. It is a scheduled breaking change. Ignoring it means your code breaks on the next library update — often at the worst possible moment.

**Generalise:** Plotly works in Streamlit, Dash, Jupyter, and standalone HTML. The same `go.Figure()` renders everywhere. This portability makes it the standard for interactive scientific visualisation in Python.

---
# PART B — Dashboard Concepts

## Section 5 — Dashboard Components Explained

### The sidebar — separating controls from content

The sidebar (`st.sidebar`) holds all the controls: dataset selector, sample size selector, and three threshold sliders. Moving controls to the sidebar follows a key UI design principle:

**Separate what the user adjusts (controls) from what the user reads (content).**

If the sliders were inline with the charts, every scroll would accidentally trigger a rerun. The sidebar keeps controls always visible and always accessible without interrupting the reading flow.

### Sliders — live threshold adjustment

The three sliders allow the student to answer important questions without rerunning any experiments:

- *What if I raise the faithfulness threshold to 0.85?* → move the slider → all gate verdicts instantly update
- *What if the recall lift target is +20pp instead of +15pp?* → move the slider → target line in ablation chart moves
- *What threshold makes V2 get reliably blocked?* → experiment interactively with the data

This is the dashboard's core value — exploring parameter sensitivity without touching the experiment code.

### Metric cards — KPI display

`st.metric()` displays a Key Performance Indicator card:
```python
st.metric(
    label="Faithfulness (with gate)",
    value="0.833",
    delta="+0.000"   # green if positive, red if negative
)
```

Cards show the most important numbers at a glance, before the viewer reads the detailed charts.

### Condition colour cards — HTML in Streamlit

Section 0 of the dashboard shows four coloured cards, one per condition. These use raw HTML injected via `st.markdown(..., unsafe_allow_html=True)`.

Streamlit normally escapes HTML for security. `unsafe_allow_html=True` bypasses this. The name "unsafe" refers to the theoretical risk of injecting malicious HTML — in our own dashboard with our own data, this is safe.

### `st.stop()` — graceful error handling

If the metrics file does not exist (step04 has not been run), the dashboard:
1. Shows an error message with `st.error()`
2. Calls `st.stop()` — halts script execution at that point

Without `st.stop()`, the script would continue running and crash with a `KeyError` or `NoneType` error — showing a confusing Python traceback to the user. `st.stop()` gives a clean, understandable error message instead.

### The drill-down table — from aggregate to individual

Section 5 of the dashboard shows individual question results. This is the "drill-down" pattern:
- Section 1 shows aggregate gate metrics (mean faithfulness across all questions)
- Section 5 lets you select a condition and see the per-question breakdown

This is essential for debugging. If the aggregate faithfulness is 0.833, which specific question scored 0.50? The drill-down answers this in one click.

In [ ]:
# Demonstrate the key Streamlit patterns in pure Python
# (without Streamlit running — conceptual demonstration)

import json
from pathlib import Path

# Load real data
with open(Path("results") / "metrics_hotpotqa_3.json") as f:
    metrics = json.load(f)
with open(Path("results") / "raw_results_hotpotqa_3.json") as f:
    raw = json.load(f)

print("=" * 58)
print("DASHBOARD COMPONENT DEMONSTRATION (non-Streamlit)")
print("=" * 58)

# ── Metric cards (what st.metric() displays) ──────────────────────
print("\n1. Metric cards (st.metric):")
e1 = metrics.get("exp1", {})
ng = e1.get("no_gate", {})
wg = e1.get("with_gate", {})

gate_rate = e1.get("gate_trigger_rate", 0)
latency   = e1.get("latency_overhead_s", 0)
delta_f   = wg.get("faithfulness",0) - ng.get("faithfulness",0)

cards = [
    ("Faithfulness (no gate)",  f"{ng.get('faithfulness',0):.3f}", None),
    ("Faithfulness (w/ gate)",  f"{wg.get('faithfulness',0):.3f}", f"{delta_f:+.3f}"),
    ("Gate trigger rate",       f"{gate_rate:.0%}",                None),
    ("Latency overhead",        f"+{latency:.2f}s/query",          None),
]
for label, value, delta in cards:
    delta_str = f" (Δ {delta})" if delta else ""
    print(f"  ┌─ {label} ─────")
    print(f"  │  {value}{delta_str}")
    print(f"  └─────────────")

# ── Sidebar slider effect ─────────────────────────────────────────
print("\n2. Slider effect on gate verdict:")
e3 = metrics.get("exp3", {})
v1 = e3.get("v1_faithfulness", 0)
v2 = e3.get("v2_faithfulness", 0)

print(f"   V1={v1:.3f}, V2={v2:.3f}")
for threshold in [0.70, 0.75, 0.80, 0.85, 0.90]:
    v1_pass  = v1 >= threshold
    v2_block = v2 <  threshold
    effective = v1_pass and v2_block
    print(f"   Threshold={threshold:.2f}: V1={'PASS' if v1_pass else 'FAIL':4s}  "
          f"V2={'BLOCKED' if v2_block else 'PASSED':7s}  "
          f"Gate={'effective ✓' if effective else 'NOT effective ✗'}")

print()
print("   The dashboard slider lets the user see this table interactively.")
print("   No code changes — just move the slider.")

# ── Drill-down table ──────────────────────────────────────────────
print("\n3. Drill-down table (per-question, Condition D):")
conds = raw.get("exp2", {}).get("conditions", {})
d_results = list(conds.values())[3] if len(conds) >= 4 else []

print(f"   {'Q#':>3} {'Faithfulness':>14} {'F1':>6} {'Recall':>8}  Answer")
print(f"   {'-'*55}")
for i, rec in enumerate(d_results):
    faith  = rec.get("faithfulness", 0)
    f1     = rec.get("f1", 0)
    recall = rec.get("context_recall", 0)
    answer = rec.get("answer", "")[:25]
    flag   = " ← gate fires" if faith < 0.75 else ""
    print(f"   {i+1:>3} {faith:>14.3f} {f1:>6.3f} {recall:>8.3f}  {answer}{flag}")

### 📌 Retain: Dashboard design patterns

**Sidebar pattern:** Controls (inputs) in sidebar, results (outputs) in main area. This is standard in data dashboards — Tableau, Power BI, Metabase all use this layout.

**Metric card pattern:** Show the 3–5 most important numbers at the top. The viewer reads these first. Charts provide the detail.

**Drill-down pattern:** Aggregate → detail. Start with averages, allow the user to inspect individual records. Every dashboard that analyses multi-record data should have this.

**Graceful error pattern:**
```python
if not data:
    st.error("Data not found. Run step04 first.")
    st.stop()   # clean halt — no traceback shown to user
```

**Generalise:** These four patterns (sidebar controls, metric cards, charts, drill-down) appear in virtually every production dashboard — from startup analytics tools to enterprise BI platforms. Learning them in Streamlit transfers directly to building dashboards in any framework.

---
## Section 6 — The Six Dashboard Sections Explained

### Section 0 — System Overview

Four coloured cards, one per retrieval condition. The colour matches the Wong palette used in all step05 figures — Condition D is always dark blue. A user who has seen the ablation bar chart immediately recognises Condition D in the dashboard.

**HTML card pattern:**
```python
st.markdown(
    f"<div style='border-left:4px solid {colour}; padding:12px;'>"
    f"<b>Condition {letter}</b><br>{name}"
    f"</div>",
    unsafe_allow_html=True
)
```

### Section 1 — Retry Gate

Five metric cards + a paired bar chart + an info box. The threshold line in the chart updates when the sidebar slider moves — because the slider value is a Python variable passed directly to `fig.add_hline(y=faith_thresh)`.

### Section 2 — Ablation

A data table + recall lift metric + a grouped bar chart. The target line updates with the recall lift slider. The table shows all four conditions side by side — easier to compare than separate charts.

### Section 3 — CI/CD Gate

Three metric cards + a horizontal gauge chart + a pass/fail verdict box. The verdict (`st.success` vs `st.warning`) updates when the faithfulness threshold slider moves — so the student can find the exact threshold at which the gate becomes effective.

### Section 4 — Literature Comparison

A table with arXiv links as markdown. The links are clickable in the browser — the viewer can go directly from the dashboard to the primary paper.

### Section 5 — Per-Question Drill-Down

A dropdown to select condition + a per-question table + automatic highlighting of questions where faithfulness < threshold. This is where a researcher would look after seeing an unexpected aggregate score — "which question caused the low average?"

---
# PART C — Running Step06

## Section 7 — Launching and Exploring the Dashboard

In [ ]:
# Step-by-step launch instructions and exploration guide
from pathlib import Path

print("=" * 58)
print("HOW TO LAUNCH THE DASHBOARD")
print("=" * 58)
print()
print("Step 1: Open a terminal in the project root")
print("  cd /run/media/bala/HDD/Projects/scigraphagent-benchmark")
print()
print("Step 2: Activate the virtual environment")
print("  source .venv/bin/activate")
print()
print("Step 3: Launch Streamlit")
print("  streamlit run step06_dashboard.py")
print()
print("Step 4: Open your browser")
print("  http://localhost:8501")
print()
print("Step 5: Stop the dashboard")
print("  Press Ctrl+C in the terminal")
print()
print("=" * 58)
print("EXPLORATION GUIDE — what to try")
print("=" * 58)
print()
print("1. Sidebar — Dataset/N:")
print("   Change dataset from hotpotqa to musique")
print("   → If no metrics for that dataset, see the error message and st.stop()")
print()
print("2. Sidebar — Faithfulness threshold slider:")
print("   Move from 0.75 to 0.90")
print("   → Watch Section 3 gate verdict change from WARNING to SUCCESS")
print("   → The threshold line in Section 1 chart moves in real time")
print()
print("3. Sidebar — Recall lift target slider:")
print("   Move from 0.15 to 0.05")
print("   → The target line in Section 2 ablation chart moves down")
print("   → At +5pp target, D=B so target is instantly met at n=3")
print()
print("4. Section 1 — Retry gate chart:")
print("   Hover over bars → exact values appear in tooltip")
print("   Click 'No gate' in legend → hides that trace")
print()
print("5. Section 5 — Drill-down table:")
print("   Select 'A: No retrieval' from dropdown")
print("   → See which specific questions scored 0.00 faithfulness")
print("   → Compare to Condition D")
print()
print("6. Section 4 — Literature table:")
print("   Click an arXiv link → paper opens in new tab")

# Verify the dashboard file is present and non-empty
dash_path = Path("step06_dashboard.py")
print()
print("=" * 58)
if dash_path.exists():
    lines = dash_path.read_text().count("\n")
    size  = dash_path.stat().st_size // 1024
    print(f"✓ step06_dashboard.py: {lines} lines, {size} KB")
    print(f"✓ Ready to launch with: streamlit run step06_dashboard.py")
else:
    print("✗ step06_dashboard.py not found")

---
## Section 8 — Honest Interpretation at n=3

The dashboard shows the same data as the step04 metrics and step05 figures. The n=3 limitations are the same — but the dashboard makes them easier to explore.

### What the dashboard reveals about n=3 that figures do not

**Threshold sensitivity (Section 3):**
By moving the faithfulness threshold slider from 0.75 to 0.83, you can see V2 (0.833) flip from PASSED to BLOCKED. This means the CI/CD gate works — but only if you happen to set the threshold at exactly the right value. At n=3, the threshold needs fine-tuning that is not robust. At n=50, V2 would settle at a reliably lower value and be blocked at any threshold from 0.65 to 0.80.

**Drill-down reveals the coverage problem (Section 5):**
Select Condition D and look at the per-question F1 scores. All three are 0.000 — but the Context Recall column shows which questions actually have the gold answer in the retrieved context. This reveals: the retrieval is working (recall > 0 for some questions), but the generator is not correctly extracting the answer even when it is present. This is a synthesis failure, not a retrieval failure.

**Literature comparison (Section 4):**
Our F1=0.000 in the table next to Self-RAG F1=0.450 looks alarming. But clicking the arXiv link for Self-RAG and reading Section 4 of their paper shows they evaluated on n=2,837 questions with gold context — not retrieved context. Our 0.000 is not comparable. The dashboard makes this transparency easy.

### What to expect when you run n=50

The dashboard will automatically update when `metrics_hotpotqa_50.json` exists. Select n=50 in the sidebar dropdown and all six sections update instantly — no code changes needed. This is the dashboard's main value for the paper run.

In [ ]:
# Verify step06 code quality — check for the deprecation fix
from pathlib import Path

print("=" * 55)
print("STEP06 CODE QUALITY VERIFICATION")
print("=" * 55)
print()

dash_path = Path("step06_dashboard.py")
if not dash_path.exists():
    print("✗ step06_dashboard.py not found")
else:
    content = dash_path.read_text()
    lines   = content.split("\n")

    # Check deprecation fix was applied
    old_param  = "use_container_width"
    new_param  = "width='stretch'"
    old_count  = content.count(old_param)
    new_count  = content.count(new_param)

    print(f"File stats:")
    print(f"  Lines : {len(lines)}")
    print(f"  Size  : {dash_path.stat().st_size // 1024} KB")
    print()
    print(f"Deprecation fix:")
    if old_count == 0:
        print(f"  ✓ use_container_width: 0 occurrences (removed)")
    else:
        print(f"  ✗ use_container_width: {old_count} occurrences remaining")
        print(f"    Fix: Ctrl+H → replace use_container_width=True with width='stretch'")
    print(f"  ✓ width='stretch'     : {new_count} occurrences")
    print()

    # Check key components are present
    components = [
        ("@st.cache_data",    "Caching decorator"),
        ("st.sidebar",        "Sidebar"),
        ("st.slider",         "Sliders"),
        ("st.metric",         "Metric cards"),
        ("st.plotly_chart",   "Plotly charts"),
        ("st.dataframe",      "Data tables"),
        ("st.stop()",         "Graceful error handling"),
        ("go.Figure()",       "Plotly figure creation"),
        ("unsafe_allow_html", "HTML condition cards"),
        ("arXiv",             "Literature links"),
    ]
    print("Key components:")
    for pattern, label in components:
        present = pattern in content
        print(f"  {'✓' if present else '✗'}  {label:<25} ({pattern})")

---
## Section 9 — Pre-Commit Verification and Commit

In [ ]:
import subprocess
from pathlib import Path

print("=" * 55)
print("PRE-COMMIT VERIFICATION")
print("=" * 55)
print()

checks = []
def check(desc, ok, detail=""):
    checks.append(ok)
    print(f"  {'✓' if ok else '✗'}  {desc}")
    if detail: print(f"       {detail}")

# Script file
dash_path = Path("step06_dashboard.py")
check("step06_dashboard.py exists", dash_path.exists())

if dash_path.exists():
    content = dash_path.read_text()

    # Deprecation fix
    check("Deprecation fixed (no use_container_width)",
          "use_container_width" not in content,
          "Run: Ctrl+H → replace use_container_width=True with width='stretch'")

    # Key components
    check("@st.cache_data present",    "@st.cache_data"    in content)
    check("st.slider present",         "st.slider"         in content)
    check("st.metric present",         "st.metric"         in content)
    check("st.plotly_chart present",   "st.plotly_chart"   in content)
    check("st.stop() present",         "st.stop()"         in content)
    check("go.Figure() present",       "go.Figure()"       in content)
    check("6 sections present",
          all(f"SECTION {i}" in content or
              f"Section {i}" in content or
              f"· Experiment {i}" in content or
              f"· System" in content
              for i in range(7)))

# Input files exist
check("metrics_hotpotqa_3.json exists",
      (Path("results") / "metrics_hotpotqa_3.json").exists())
check("raw_results_hotpotqa_3.json exists",
      (Path("results") / "raw_results_hotpotqa_3.json").exists())

# Git status
print()
print("Git status:")
result = subprocess.run("git status --short", shell=True,
                        capture_output=True, text=True)
for line in result.stdout.strip().split("\n"):
    if line.strip(): print(f"  {line}")

passed = sum(checks)
total  = len(checks)
print()
print(f"{'✓' if passed == total else '✗'} {passed}/{total} checks passed")
print()
print("Commit command:")
print("  git add step06_dashboard.py notebooks/06_dashboard.ipynb")
print('  git commit -m "Add step06 and Notebook 06: interactive Streamlit dashboard"')
print("  git push")

---
## Summary — What We Learned and How to Apply It

### Engineering constraints (Part A)

| Concept | Rule | Generalise |
|---|---|---|
| Dashboard cost | $0 — reads JSON only | Separate analysis from presentation |
| Streamlit rerun | Every widget interaction reruns the whole script | Cache all expensive operations |
| @st.cache_data | Caches function return value by arguments | Any slow function called repeatedly |
| Deprecation warnings | Fix immediately — scheduled breaking change | Any library warning in any project |
| Plotly in Streamlit | `st.plotly_chart(fig, width='stretch')` | Any interactive chart in Streamlit |

### Dashboard concepts (Part B)

| Pattern | Streamlit code | Generalise |
|---|---|---|
| Sidebar controls | `with st.sidebar: st.slider(...)` | All dashboards — separate input from output |
| Metric card | `st.metric(label, value, delta)` | KPI display in any analytics tool |
| Graceful error | `st.error(...); st.stop()` | Any app with optional data dependencies |
| HTML cards | `st.markdown(..., unsafe_allow_html=True)` | Custom styled components |
| Drill-down | Dropdown → filter → `st.dataframe()` | Any aggregate → detail pattern |

### Engineering decisions in step06 — with reasoning

| Decision | Choice | Alternative | Reason |
|---|---|---|---|
| Framework | Streamlit | Flask, Dash | Hours not weeks to build; pip-installable |
| Charts | Plotly | matplotlib | Interactive hover/zoom in browser |
| Caching | @st.cache_data | No caching | Prevents file re-read on every slider move |
| Colour palette | Wong 2011 | matplotlib defaults | Consistent with step05, colour-blind safe |
| Sidebar | Controls in sidebar | Inline with content | Standard UX: separate input from output |
| st.stop() | On missing data | Let it crash | Clean user-facing error, no Python traceback |
| Threshold sliders | Sidebar, adjustable | Hardcoded | Student explores parameter sensitivity |

### The complete pipeline — end to end

```
step01: HuggingFace → data/hotpotqa_sample_n.json         (0 API calls)
step02: data/ → index/chroma/ + knowledge graph             (0 API calls)
step03: index/ + graph → results/raw_results.json           (~45 API calls per n=3 run)
step04: raw_results.json → results/metrics.json             (0 API calls)
step05: metrics.json → results/figures/*.png                (0 API calls)
step06: metrics.json + raw.json → http://localhost:8501     (0 API calls)
```

Only step03 makes API calls. Everything else is free to re-run.

### What remains

- **Section 10 of Notebook 03:** Run `step03.main()` from inside the notebook — deferred to tomorrow (fresh Groq TPD budget)
- **n=50 development run:** Two days of free Groq quota, starting tomorrow
- **n=1000 paper run:** Google Colab Pro recommended (~$10), ~$18 API cost
- **Paper write-up:** Results at n=1000 fill in the TBD cells in the literature comparison table

---
*Notebook 06 complete — final notebook in the series. Move to `notebooks/`, run all cells, commit both files together.*